In [ ]:
!which python

# 

## За основу оберток для моделей взял классы из: 
https://github.com/tim-ponomarev/hybrid-rag/blob/main/src/rerank.py


In [ ]:
!pip install torch 


In [ ]:
!pip install pandas pyarrow beautifulsoup4 langchain-community ipywidgets lxml
!pip install -U "sentence-transformers[onnx]"

In [ ]:
!pip install FlagEmbedding
!pip install rank-bm25 pymorphy3

In [ ]:
!pip install faiss-gpu 

In [ ]:
#!pip install faiss

In [ ]:
import pandas as pd

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
import re
import lxml
from bs4 import NavigableString, Tag

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS, VectorStore
from langchain_core.documents import Document


In [ ]:
from nltk.tokenize import word_tokenize
import nltk
from rank_bm25 import BM25Okapi
from dataclasses import dataclass
from pymorphy3 import MorphAnalyzer
import numpy as np

# Предзагрузка моделей

In [ ]:

model = SentenceTransformer('BAAI/bge-m3')
model_2= SentenceTransformer('BAAI/bge-reranker-v2-m3')

num_params = sum(p.numel() for p in model.parameters())
print(f'BAAI/bge-m3: {num_params / 1e6:.1f}M параметров')
num_params = sum(p.numel() for p in model_2.parameters())
print(f'BAAI/bge-reranker-v2-m3: {num_params / 1e6:.1f}M параметров')




## Загружаем данные

In [ ]:
articles_data = pd.read_feather('articles.f')
calibration_data = pd.read_feather('calibration.f')
test_data = pd.read_feather('test.f')



In [ ]:
calibration_data.head(25)

In [ ]:
articles_data.head()

In [ ]:
test_data.head()

## Очищаем html:
Удаляем пустые строки, скрипты, пробелы, ненужные блоки.
Нормализуем и применяем к нашим body.


In [ ]:
BLOCK_TAGS = {
    "p", "div",
    "h1", "h2", "h3", "h4", "h5", "h6",
    "li", "ul", "ol",
    "table", "tr",
    "section", "article",
    "details", "summary",
    "blockquote",
    "br"
}


def cleaner(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "svg", "noscript"]):
        tag.decompose()

    result = []

    def walk(node):
        if isinstance(node, NavigableString):
            text = str(node)
            if text.strip():
                result.append(text)

        elif isinstance(node, Tag):
            if node.name == "br":
                result.append("\n")
                return

            for child in node.children:
                walk(child)

            if node.name in BLOCK_TAGS:
                result.append("\n")

    walk(soup)

    text = "".join(result)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"\n([.,;:!?])", r"\1", text)

    return text.strip()

In [ ]:
articles_data['clean_body'] = articles_data['body'].apply(cleaner)

# Смотрим базовую статистику по длине тел.
# Можем заметить, что в данных есть существенный разброс.


In [ ]:
std = np.std(articles_data['clean_body'].str.len())
print(std)
mean = np.mean(articles_data['clean_body'].str.len())
print(mean)
mx = np.max(articles_data['clean_body'].str.len())
print(mx)

In [ ]:
calibration_data.isna().any()

In [ ]:
articles_data.isna().any()

In [ ]:
articles_data['body_len'] = articles_data['clean_body'].str.len()

In [ ]:
articles_data['body_len'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.95])

# Посколько статья на 506000 символов - это не норм, буду считать ее "выбросом" и удалю из группы.

# По хорошему, ее нужно разбить на n статей, с разными id и title, поскольку информации в ней довольно много.

In [ ]:
articles_data[articles_data['clean_body'].str.len() == max]

In [ ]:
calibration_data[calibration_data['ground_truth'].str.contains('2924', na=False)]

In [ ]:
articles_data.drop(index=245, inplace=True)

## При использовании эмбеддинг модели, длинные тексты могут терять смысл, поэтому лучше будет разбить их.
## Приведем к среднему.


In [ ]:
def split_titles_with_parts(df, max_len=5000):
    new_rows = []
    for _, row in df.iterrows():
        text = row['clean_body']
        
        if len(text) <= max_len:
            row['part'] = 1
            row['total_parts'] = 1
            new_rows.append(row)
        else:
            parts = [text[i:i+max_len] for i in range(0, len(text), max_len)]
            for i, part in enumerate(parts, 1):
                new_row = row.copy()
                new_row['clean_body'] = part
                new_row['part'] = i
                new_row['total_parts'] = len(parts)
                new_rows.append(new_row)
    
    return pd.DataFrame(new_rows)

In [ ]:
articles_data= split_titles_with_parts(articles_data, max_len=5000)

In [ ]:
articles_data

# Так же, после тестов, решил подмешивать заголовок к самому телу, ибо отдельная работа с заголовками не показала хороших результатов.

In [ ]:
articles_data['clean_body_title'] = articles_data['title'] + ' ' + articles_data['clean_body']
articles_data.drop(columns=['body'], inplace=True)

# Основа классов была взяла из указанного в начале репозитория.

# Некоторые методы и аттрибуты класса были подправлены, изначально он работал с qdrant.

# Так как после разбиения документов на чанки одна статья может представляться несколькими чанками с одинаковым article_id, в первые top_k результатов могут попадать несколько   чанков одной и той же статьи. Поэтому мы извлекаем больше кандидатов (k > top_k), затем удаляем дубликаты по article_id, оставляя наиболее релевантный чанк каждой статьи.

In [ ]:
@dataclass
class RetrievedDoc:
    doc_id: str
    text: str
    score: float
    source: str  # "bm25" / "dense" / "hybrid"
    metadata: dict | None = None



In [ ]:

class DenseRetriever:
    def __init__(
        self,
        vector_store: VectorStore,
        distance_strat: str  = 'cosine'
    ):

        self.distance_strat = distance_strat
        self.vector_store = vector_store


    def search(self, query: str, top_k: int = 50) -> list[RetrievedDoc]:
        hits = self.vector_store.similarity_search_with_relevance_scores(query=query, k=top_k*3) #очевидно что *3 - может упасть, если в топ попадет достаточно много
                                                                                                 # одинаковых id, нужно либо смотреть максимальное количество частей
                                                                                                 # для 1 статьи, либо динамически добирать id.
        best_docs = {}

        for doc, score in hits:
            article_id = str(doc.metadata["id"])

            retrieved = RetrievedDoc(
                doc_id=article_id,
                text=doc.page_content,
                score=float(score),
                source="dense",
                metadata=doc.metadata,
            )

            if (
                article_id not in best_docs
                or retrieved.score > best_docs[article_id].score
            ):
                best_docs[article_id] = retrieved

        return sorted(
            best_docs.values(),
            key=lambda x: x.score,
            reverse=True
        )[:top_k]

# Инициализируем модель для эмбеддмнгов, и заполняем vector_store



In [ ]:
embeds = HuggingFaceEmbeddings(model_name='BAAI/bge-m3', encode_kwargs={
        "batch_size": 4,
        "normalize_embeddings": True})

In [ ]:
docs_for_faiss = []
for row in articles_data.itertuples():
  docs_for_faiss.append(Document(page_content=row.clean_body_title, metadata={'id': row.article_id, 'part': row.part}))

In [ ]:
vector_store = FAISS.from_documents(docs_for_faiss, embedding=embeds)

# Создаем объект ретривера

In [ ]:
retriever_dense = DenseRetriever(vector_store=vector_store)

# Во время тестов, считал разные метрики, например Recall@k - помогал понять, достаточно ли хорошо работает эмбеддер, и что именно нужно улучшать.


In [ ]:
def recall_at_k(relevant_ids, retrieved_ids, k=10):
    relevant = set(relevant_ids)
    retrieved = set(retrieved_ids[:k])

    if len(relevant) == 0:
        return 0.0

    return len(relevant & retrieved) / len(relevant)

In [ ]:
def mean_recall_at_k(k):
    s = []
    for row in calibration_data.itertuples():
        query = row.query_text
        retrieves = []
        truth = row.ground_truth.split()
        for r in retriever_dense.search(query, k):
            retrieves.append(str(r.doc_id))
        rcal = recall_at_k(truth, retrieves, k)
        s.append(rcal)
    return sum(s)/len(s)
        
        

In [ ]:
print(mean_recall_at_k(30))
print(mean_recall_at_k(20))
print(mean_recall_at_k(10))

# Определяем целевую метрику


In [ ]:
def average_precision_at_10(relevant_ids, retrieved_ids):
    relevant_set = set(relevant_ids)
    retrieved_ids = retrieved_ids[:10]
    if not relevant_set:
        return 0.0
    precision_sum = 0.0
    relevant_found = 0
    for i, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_set:
            relevant_found += 1
            precision_sum += relevant_found / i
    return precision_sum / min(len(relevant_set), 10)


def map_at_10(test_queries):
    ap_scores = []
    for test in test_queries:
        ap_scores.append(average_precision_at_10(test['relevant_ids'], test['retrieved_ids']))
    return sum(ap_scores) / len(ap_scores) if ap_scores else 0.0

In [ ]:
from FlagEmbedding import FlagReranker

In [ ]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True, trust_remote_code=True) #BAAI/bge-reranker-v2-m3

# Создаем итоговую функцию всего пайплайна!


In [ ]:
def pipeline_retrieve(
    query: str      ) -> list[str]:
    pairs = []
    id_docs = []

    candidates = retriever_dense.search(query, 15)

    for doc in candidates:
        pairs.append([query, doc.text])
        id_docs.append(doc.doc_id)

    if not pairs:
        return []
    
    scores = reranker.compute_score(
        pairs,
        normalize=True,
        batch_size=4
    )
    ranked = sorted(
        zip(id_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc_id for doc_id, _ in ranked[:10]]

# Тестируем

In [ ]:
pipeline_retrieve('Как передать товар через службу авито')

# Считаем метрику на тестовом наборе:

In [ ]:
itg = []
for row in calibration_data.itertuples():
    ground_truth= row.ground_truth.split()
    reranked = pipeline_retrieve(row.query_text)
    itg.append(
    {
        'relevant_ids': ground_truth,
        'retrieved_ids': reranked
                
    }
    
    )

In [ ]:
print(f'MAP@10: {map_at_10(itg)}')

# Получаем ответ

In [ ]:
test_data['answer'] = test_data['query_text'].apply(lambda x: " ".join(pipeline_retrieve(x)))

In [ ]:
test_data.head()

In [ ]:
test_data[['query_id', 'answer']].to_csv('answer.csv', index=False)